<div align="center">

<a href="https://colab.research.google.com/github/utkukose/philosophy-of-ai-NB-lecture/blob/main/weeks/week-02/NB02_turing_test.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Week 02: Can Machines Think? The Turing Test and Its Critics

**Philosophy of Artificial Intelligence (11118BLG003)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

*This course is updated in line with current developments in the field. Last update: September 2026.*

</div>

## How to use this notebook

This notebook is a self-contained version of the week: the lecture text comes first, followed by the hands-on lab. Run the cells in order with Shift+Enter. Exercise cells start with `None` placeholders and print feedback through `check(...)`. Reference solutions sit in collapsed cells: open them only after a genuine attempt. The interactive lab and the self-assessment are available at <https://utkukose.github.io/philosophy-of-ai-NB-lecture/weeks/week-02/lab.html>.

## Overview

In 1950, Alan Turing replaced the question of whether machines can think with a game of imitation. This week reconstructs his proposal and his replies to objections, examines critiques that target the sufficiency of behaviour, and interprets recent experiments in which language models were judged to be human [1, 2, 3].

## Learning outcomes

By the end of the week, students are expected to describe the imitation game and Turing's prediction, to evaluate at least three of the objections he considered, to explain why Blockhead and subcognitive questions challenge behavioural tests, and to interpret the results of recent three-party Turing tests with appropriate statistical care.

## Replacing the question

Turing found the question of whether machines can think too vague to answer, so he proposed to replace it with a game [1]. An interrogator communicates in writing with two hidden witnesses, one human and one machine, and must decide which is which. If machines play the game so well that interrogators do no better than chance, the original question loses its point. Turing also made a prediction: that by about the end of the twentieth century, an average interrogator would have no more than a 70 percent chance of making the right identification after five minutes of questioning.

The proposal is behavioural and deliberately so. It sets aside the appearance of the machine and the material it is made of and focuses on verbal behaviour, which Turing treated as a sufficient basis for attributing thought, just as people attribute thought to one another.

## Objections and replies

Turing anticipated nine objections to his view: the theological objection, the heads-in-the-sand objection, the mathematical objection, the argument from consciousness, arguments from various disabilities, Lady Lovelace's objection, the argument from the continuity of the nervous system, the argument from the informality of behaviour and the argument from extrasensory perception [1]. Three of them return later in the course. The mathematical objection draws on Gödel's theorem and is examined in Week 9. The argument from consciousness, which holds that a machine must feel what it writes, is examined in Weeks 10 and 11. Lady Lovelace's objection, that a machine can only do what it is ordered to do, is examined in Week 13. To the argument from consciousness, Turing replied that it leads to solipsism, since the only way to be sure that a person thinks would be to be that person. Saygin, Cicekli and Akman reviewed the first fifty years of debate about the test and its variants [4].

## Is behaviour enough?

Critics have questioned whether passing the test is sufficient for thought. Block imagined a machine, later called Blockhead, that stores a sensible reply for every possible conversation up to the length of the test [2]. Such a lookup table would pass, yet it would be as unintelligent as a jukebox. Block concluded that intelligence depends on how behaviour is produced, not only on the behaviour itself, a position he called psychologism. The practical impossibility of the table does not answer the point, since the argument concerns what the test measures, not what can be built.

French argued that the test is too demanding in a different way [5]. Subcognitive questions, such as asking how good a made-up word would be as a name for a breakfast cereal, probe associations that only an entity with a human life history would share. The test may therefore detect humanness rather than intelligence. Weizenbaum's ELIZA showed a third problem from the opposite direction: A simple program that transformed user sentences with pattern-matching rules led some users to attribute understanding to it [6]. Human judges are not neutral instruments.

## The test in the age of language models

Jones and Bergen ran randomised, controlled and pre-registered three-party Turing tests in which interrogators held five-minute conversations simultaneously with a human and a machine [3]. When prompted to adopt a humanlike persona, GPT-4.5 was judged to be the human in 73 percent of the games, significantly more often than the real human participants. LLaMa-3.1-405B with the same prompt was judged human 56 percent of the time, while ELIZA and GPT-4o without a persona reached 23 and 21 percent. Figure 2.1 shows these reported rates.

The results meet Turing's operational criterion in its standard form. Whether they show that the systems think is exactly the question that Block and French raised. The results also depended strongly on the persona prompt, which suggests that the test measures skill at a particular social performance. The lab asks students to act as interrogators, to compute whether their own accuracy exceeds chance and to estimate the size of Blockhead's table.

> **Pause and reflect.** If a system passes the test only with a carefully written persona, who or what has passed the test: the model, the prompt writer or the combination?

![Figure 2.1](https://raw.githubusercontent.com/utkukose/philosophy-of-ai-NB-lecture/main/weeks/week-02/figures/w02_fig1.png)

*Figure 2.1. Share of games in which each system was judged to be the human in the three-party Turing tests reported by Jones and Bergen; the dashed line marks chance [3].*

# Hands-on lab

The notebook implements a small ELIZA-style program, models interrogators with signal detection theory to relate accuracy to Turing's 70 percent criterion, tests judge accuracy against chance and computes the size of Blockhead's table [1, 2, 6].

In [ ]:
# Setup: common libraries and the self-check helper used in the exercises.
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 2026

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = bool(np.allclose(np.asarray(value, dtype=float), np.asarray(expected, dtype=float), atol=tol))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] " + ("Correct." if ok else "Not yet. Revisit the explanation above and try again."))

## 1. A minimal ELIZA

The program matches patterns in the input and fills templates with reflected fragments of the user's words. It stores no knowledge about the world.

In [ ]:
import re
from scipy.stats import binomtest, norm

REFLECT = {"i": "you", "me": "you", "my": "your", "am": "are", "you": "I", "your": "my", "i'm": "you are"}
RULES = [
    (r"i need (.*)", ["Why do you need {0}?", "Would it really help you to get {0}?"]),
    (r"i am (.*)", ["How long have you been {0}?", "Why do you think you are {0}?"]),
    (r"because (.*)", ["Is that the real reason?", "What other reasons come to mind?"]),
    (r"(.*)\bmother\b(.*)", ["Tell me more about your family."]),
    (r"(.*)", ["Please tell me more.", "How does that make you feel?", "I see. Go on."]),
]
rng = np.random.default_rng(SEED)

def reflect(fragment):
    return " ".join(REFLECT.get(w, w) for w in fragment.split())

def eliza(sentence):
    s = sentence.lower().strip(" .!?")
    for pattern, answers in RULES:
        m = re.match(pattern, s)
        if m:
            return answers[rng.integers(len(answers))].format(*[reflect(g) for g in m.groups()])

for line in ["I need a holiday.", "I am worried about my exams.", "Because my mother called.", "The weather is nice."]:
    print(f"> {line}\n  {eliza(line)}")

## 2. Interrogators as signal detectors

In a two-alternative version of the game, an interrogator with sensitivity d' picks the human correctly with probability Phi(d' / sqrt(2)), where Phi is the standard normal distribution function.

In [ ]:
d = np.linspace(0, 3, 100)
plt.plot(d, norm.cdf(d / np.sqrt(2)))
plt.axhline(0.7, color="k", ls="--", lw=0.8); plt.text(0.05, 0.71, "Turing's 70 percent criterion")
plt.xlabel("interrogator sensitivity d'"); plt.ylabel("probability of a correct identification"); plt.show()

### Your turn, exercise 1

Compute the sensitivity d' at which an interrogator is correct exactly 70 percent of the time.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _dprime_reference():
    return float(np.sqrt(2) * norm.ppf(0.7))

In [ ]:
d_prime_70 = None  # your computation
check("Exercise 1", d_prime_70, _dprime_reference())

## 3. Is a judge better than chance?

Suppose that, in a hypothetical study, interrogators identified the human correctly in 60 of 100 games.

### Your turn, exercise 2

Compute the one-sided p-value for 60 correct identifications out of 100 under the hypothesis of guessing (p = 0.5). Use `binomtest(..., alternative='greater')`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _pvalue_reference():
    return binomtest(60, 100, 0.5, alternative="greater").pvalue

In [ ]:
p_value = None  # your computation
check("Exercise 2", p_value, _pvalue_reference())

## 4. The size of Blockhead

In [ ]:
for vocab, words, turns in [(10_000, 10, 5), (10_000, 10, 10), (50_000, 20, 20)]:
    exponent = turns * words * np.log10(vocab)
    print(f"vocabulary {vocab}, {words} words per message, {turns} messages: about 10^{exponent:.0f} histories")

Block's argument does not depend on whether such a table can be built. It asks whether passing the test is sufficient for intelligence in principle [2].

## Self-assessment and reflection

Take the self-assessment in the interactive lab: <https://utkukose.github.io/philosophy-of-ai-NB-lecture/weeks/week-02/lab.html> (tab: Check yourself). Then answer the reflection prompts:

1. How accurate were you in Part A, and what cues did you rely on? Would the same cues work against a persona-prompted model?
2. Does Blockhead refute behavioural tests of intelligence, or only show their limits? Defend your view.
3. Design a question that you believe a present language model would answer differently from a person. Explain why.

## Weekly task and submission

Write an argument analysis of about 500 words: Reconstruct Turing's argument as numbered premises and a conclusion, state which premise Block and French each attack, and evaluate whether the results of Jones and Bergen change the debate. Attach the notebook with both exercises completed.

The weekly task supports self-learning and builds a personal portfolio. When the course is followed with the instructor during an active semester, the task can be sent together with the exported learning log to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation.

## Research and report assignment (optional)

**The 2025 Turing test results and their critics.** Review the responses to the three-party Turing tests reported in 2025 and assess them with the arguments of Block and French [2, 3, 5].

This research assignment is optional and supports self-learning. When the related weeks are followed within the course during an active semester, the report can be sent to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation. Unless the assignment states otherwise, a report has 1500 to 2500 words, follows the structure of an academic paper, cites at least six scholarly or official sources in square brackets and ends with a reference list.

## References

[1] Turing, A. M. (1950). Computing machinery and intelligence. *Mind*, 59(236), 433-460. <https://doi.org/10.1093/mind/LIX.236.433>

[2] Block, N. (1981). Psychologism and behaviorism. *The Philosophical Review*, 90(1), 5-43. <https://doi.org/10.2307/2184371>

[3] Jones, C. R., & Bergen, B. K. (2025). Large language models pass the Turing test. arXiv preprint arXiv:2503.23674. <https://arxiv.org/abs/2503.23674>

[4] Saygin, A. P., Cicekli, I., & Akman, V. (2000). Turing test: 50 years later. *Minds and Machines*, 10(4), 463-518.

[5] French, R. M. (1990). Subcognition and the limits of the Turing test. *Mind*, 99(393), 53-65. <https://doi.org/10.1093/mind/XCIX.393.53>

[6] Weizenbaum, J. (1966). ELIZA: A computer program for the study of natural language communication between man and machine. *Communications of the ACM*, 9(1), 36-45. <https://doi.org/10.1145/365153.365168>